# CSCI E-89, Lecture 11
## RAG with LangChain and OpenAI
### Build the document ingestion pipeline

In [ ]:
#!pip install -q  langchain_openai
#!pip -q install langchain_chroma

In [ ]:
#!pip install pypdf

In [1]:
import os
from dotenv import load_dotenv
from langchain_classic.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

In [2]:
from langchain_chroma.vectorstores import Chroma
from langchain_classic.embeddings import OllamaEmbeddings

#### Read OpenAI_API_KEY

In [3]:
load_dotenv()

True

#### Load PDF

In [4]:
loader = PyPDFLoader("documents/How to Use LangChain and LangGraph_A Beginner-s Guide to AI Workflows.pdf")
documents = loader.load() 

#### Split the document into chunks

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(documents)

#### Create embeddings

In [6]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

#### Store in ChromaDB

In [7]:
vectorstore=Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory="./vectorstore")
print(f"Successfullyindexed{len(chunks)}chunks")

Successfullyindexed21chunks


### Build the RAG chatbot

#### Create a Chroma VectorsDB

In [8]:
from langchain_chroma import Chroma
from langchain_openai import(
    OpenAIEmbeddings,
    ChatOpenAI
)

# Load existing vector database
embeddings = OpenAIEmbeddings ( model = "text-embedding-3-small")

vectorstore=Chroma(
    persist_directory="./vectorstore",
    embedding_function=embeddings
)

#### Configure retriever

In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":4})

# Configure LLM
llm = ChatOpenAI(
    model="gpt-6.1-sol",
    temperature=1)

def ask_question ( question):  
    # Retrieve relevant documents
    docs=retriever.invoke(question)
    context="\n\n".join(
        doc.page_content for doc in docs
    )
    prompt = f"""
    Answer the question using ONLY the
    information provided in the context.
    If the answer is not available,
    say you don't know.
    Context:
    {
    context
    }
    Question:
    {
    question
    }
    Answer:
    """
    response=llm.invoke(prompt)
    print ( "\nANSWER:")
    print(response.content)
    
    print ( "\nSOURCES:")    
    for doc in docs :
        print(doc.metadata)

#### Ask a question, `exit` to exit

In [10]:
while True :
    question = input ( "\nAsk a question (or exit): ")

    if question.lower() == "exit" :
        break
    ask_question ( question)


Ask a question (or exit):  What is this document about



ANSWER:
The document is a beginner’s guide to using LangChain and LangGraph to build AI workflows, written by Manish Shivanandhan.

SOURCES:
{'page': 0, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36', 'source': 'documents/How to Use LangChain and LangGraph_ A Beginner-s Guide to AI Workflows.pdf', 'total_pages': 13, 'creationdate': '2026-11-08T03:00:35+00:00', 'title': 'How to Use LangChain and LangGraph: A Beginner’s Guide to AI Workflows', 'page_label': '1', 'moddate': '2026-11-08T03:00:35+00:00', 'producer': 'Skia/PDF m156'}
{'source': 'documents/How to Use LangChain and LangGraph_ A Beginner-s Guide to AI Workflows.pdf', 'page': 0, 'creationdate': '2026-11-08T03:00:35+00:00', 'moddate': '2026-11-08T03:00:35+00:00', 'total_pages': 13, 'page_label': '1', 'title': 'How to Use LangChain and LangGraph: A Beginner’s Guide to AI Workflows', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.3


Ask a question (or exit):  What is the meaning of life



ANSWER:
I don't know.

SOURCES:
{'creationdate': '2026-11-08T03:00:35+00:00', 'producer': 'Skia/PDF m156', 'page_label': '10', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36', 'source': 'documents/How to Use LangChain and LangGraph_ A Beginner-s Guide to AI Workflows.pdf', 'title': 'How to Use LangChain and LangGraph: A Beginner’s Guide to AI Workflows', 'moddate': '2026-11-08T03:00:35+00:00', 'page': 9, 'total_pages': 13}
{'source': 'documents/How to Use LangChain and LangGraph_A Beginner-s Guide to AI Workflows.pdf', 'title': 'How to Use LangChain and LangGraph: A Beginner’s Guide to AI Workflows', 'moddate': '2026-11-08T03:00:35+00:00', 'total_pages': 13, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36', 'creationdate': '2026-11-08T03:00:35+00:00', 'page': 9, 'page_label': '10', 'producer': 'Skia/PDF m156'}
{'total_pages': 13, 'pag


Ask a question (or exit):  exit
